# Online Retail II — Sales & Revenue Performance Analysis

**Tools:** BigQuery SQL, Google Sheets, Power BI, Looker Studio
**Full log lives here (this notebook); Notion holds the compressed summary.**

## ASK

**Business question:** Which products and regions drive the most revenue for this online retailer, how has that performance trended over time — and how much of it is being eroded by order cancellations?

## PREPARE

**Data Source**

Online Retail II — real transactions from a UK-based online retailer, 01/12/2009–09/12/2011.

- Source: UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/502/online+retail+ii
- Citation: Chen, D. (2012). Online Retail II [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- License: CC BY 4.0

**Structure**

- Delivered as two Excel sheets: "Year 2009-2010" and "Year 2010-2011"
- Columns: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country
- Combined published size: 1,067,371 rows

## PROCESS

### Loading into BigQuery

- Dataset `online_retail_analysis` created; raw tables `transactions_2009_2010` and `transactions_2010_2011` loaded as separate tables (raw layer, kept untouched).
- Both years exported from Excel as CSV UTF-8 (avoids encoding issues with £ symbols and special characters).
- Schema set manually rather than auto-detected: `Invoice` and `Customer ID` as STRING (not numeric — they're identifiers, not values to do math on), `InvoiceDate` as STRING (source format doesn't match BigQuery's TIMESTAMP parser — needs `PARSE_TIMESTAMP` on use), `Quantity` as INTEGER, `Price` as NUMERIC.

**Verification — row counts confirmed against known dataset totals:**

| Table | Row count |
|---|---|
| transactions_2009_2010 | 525,461 |
| transactions_2010_2011 | 541,910 |
| **Combined (expected)** | **1,067,371** |

Matches the published total for the full combined dataset exactly.

### Combining the two sheets

Kept raw/clean layering: raw tables stay untouched in `online_retail_analysis`; combined output goes into a separate `online_retail_combine` dataset.

Used `UNION ALL` (not plain `UNION`) — deliberately, since `UNION` silently de-duplicates and could hide a genuine duplicate-row problem without me knowing.

```sql
CREATE OR REPLACE TABLE `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined` AS

SELECT
  transactions_2009_2010.Invoice,
  transactions_2009_2010.Stockcode,
  transactions_2009_2010.Description,
  transactions_2009_2010.Quantity,
  transactions_2009_2010.InvoiceDate,
  transactions_2009_2010.Price,
  transactions_2009_2010.`Customer ID`,
  transactions_2009_2010.Country
FROM
  `project-ee6309fa-8dfb-435e-920.online_retail_analysis.transactions_2009_2010` AS transactions_2009_2010

UNION ALL

SELECT
  transactions_2010_2011.Invoice,
  transactions_2010_2011.Stockcode,
  transactions_2010_2011.Description,
  transactions_2010_2011.Quantity,
  transactions_2010_2011.InvoiceDate,
  transactions_2010_2011.Price,
  transactions_2010_2011.`Customer ID`,
  transactions_2010_2011.Country
FROM
  `project-ee6309fa-8dfb-435e-920.online_retail_analysis.transactions_2010_2011` AS transactions_2010_2011
```

**Result:** `transactions_combined` = 1,067,371 rows. Matches expected total exactly — nothing lost or duplicated by the combine.

### Null / negative / zero checks — raw vs. combined cross-check

```sql
SELECT
  'transactions_2009_2010' AS source_table,
  COUNT(*) AS total_rows,
  COUNTIF(Description IS NULL) AS null_description,
  COUNTIF(`Customer ID` IS NULL) AS null_customer_id,
  COUNTIF(Quantity < 0) AS negative_quantity,
  COUNTIF(Price = 0) AS zero_price
FROM `project-ee6309fa-8dfb-435e-920.online_retail_analysis.transactions_2009_2010`

UNION ALL

SELECT
  'transactions_2010_2011', COUNT(*),
  COUNTIF(Description IS NULL), COUNTIF(`Customer ID` IS NULL),
  COUNTIF(Quantity < 0), COUNTIF(Price = 0)
FROM `project-ee6309fa-8dfb-435e-920.online_retail_analysis.transactions_2010_2011`

UNION ALL

SELECT
  'transactions_combined', COUNT(*),
  COUNTIF(Description IS NULL), COUNTIF(`Customer ID` IS NULL),
  COUNTIF(Quantity < 0), COUNTIF(Price = 0)
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
```

**Result:**

| Source | Total rows | Null Description | Null Customer ID | Negative Quantity | Zero Price |
|---|---|---|---|---|---|
| transactions_2009_2010 | 525,461 | 2,928 | 107,927 | 12,326 | 3,687 |
| transactions_2010_2011 | 541,910 | 1,454 | 135,080 | 10,624 | 2,515 |
| transactions_combined | 1,067,371 | 4,382 | 243,007 | 22,950 | 6,202 |

Counts add up exactly across both raw tables to match the combined total on every column — confirms the `UNION ALL` combine step introduced no data loss or duplication.

**As % of total:** Null Description 0.41% | Null Customer ID 22.8% | Negative Quantity 2.15% | Zero Price 0.58%

### Cross-validating against Google Sheets (independent source check)

Spot-checked the same five metrics directly in Google Sheets (reading the original Excel file, not the CSV export used for BigQuery) on the 2009-2010 sheet:

| Metric | BigQuery | Google Sheets |
|---|---|---|
| Total rows | 525,461 | 525,461 |
| Null Description | 2,928 | 2,928 |
| Null Customer ID | 107,927 | 107,927 |
| Negative Quantity | 12,326 | 12,326 |
| Zero Price | 3,687 | 3,687 |

Exact match across every metric, from two completely independent tools reading the data two different ways. Rules out CSV parsing/import corruption as an explanation for the nulls — confirms these patterns are genuinely present in the source data.

### Verifying InvoiceDate Parsing

`InvoiceDate` loads as STRING (raw format `12/13/09 9:58` doesn't match BigQuery's TIMESTAMP auto-parser). Checked whether `PARSE_TIMESTAMP('%m/%d/%y %H:%M', InvoiceDate)` parses cleanly across the entire table before relying on it for any date math.

```sql
SELECT
  COUNTIF(SAFE.PARSE_TIMESTAMP('%m/%d/%y %H:%M', InvoiceDate) IS NULL) AS unparseable_dates
FROM
  `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
```

**Result:** 0 unparseable rows across all 1,067,371 rows. The format is fully consistent — no exceptions, no malformed dates hiding anywhere. Checklist item closed.

### Anomaly: null Description + null Customer ID + zero Price — one pattern, not three

Checked whether the three anomaly types travel together or occur independently:

```sql
SELECT COUNT(*) AS total_rows
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
WHERE
  `Customer ID` IS NULL AND Description IS NULL AND Price = 0
```

**Result:** 4,382 — matches the null-Description count exactly.

**Finding:** every row with a null Description also has a null Customer ID and Price = 0. This is one single, consistent pattern (4,382 rows, 0.41% of the dataset), not several unrelated issues. Likely non-sale entries (write-offs, damages, samples, manual adjustments) logged through the same system as real transactions rather than genuine sales.

**Decision (resolved below in "Building the Clean, Reusable Table"):** rather than exclude or footnote these rows in isolation, they became the basis for the `is_zero_value_entry` flag — see the fuller zero-price investigation later in this section, which found this 4,382-row pattern was only part of a larger, three-way zero-price picture.

### Invoice Format Anomaly: "A"-prefixed entries (not part of original checklist)

While validating that non-cancelled invoices are consistently 6 characters long, found a 6-row gap between total rows and the sum of (6-char invoices) + ("C"-prefixed invoices).

```sql
SELECT Invoice, LENGTH(Invoice) AS invoice_length
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
WHERE NOT STARTS_WITH(Invoice, 'C') AND LENGTH(Invoice) != 6
```

**Result:** 6 rows, all prefixed with "A" instead of "C", all 7 characters long.

```sql
SELECT Invoice, Description, Quantity, Price, `Customer ID`
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
WHERE STARTS_WITH(Invoice, 'A') AND LENGTH(Invoice) = 7
```

**Result:**

| Invoice | Description | Quantity | Price | Customer ID |
|---|---|---|---|---|
| A563185 | Adjust bad debt | 1 | 11062.06 | null |
| A563187 | Adjust bad debt | 1 | -11062.06 | null |
| A563186 | Adjust bad debt | 1 | -11062.06 | null |
| A516228 | Adjust bad debt | 1 | -44031.79 | null |
| A506401 | Adjust bad debt | 1 | -53594.36 | null |
| A528059 | Adjust bad debt | 1 | -38925.87 | null |

**Finding:** "A"-prefixed invoices represent bad debt write-off adjustments — a third transaction type distinct from regular sales and "C" cancellations. Customer ID is null on all six, so these can't be traced to a specific customer directly.

**Hypothesis (unconfirmed):** A563186 and A563187 are identical in amount (-11,062.06 each) — unlikely to be two coincidentally-matching separate write-offs. A563185 (+11,062.06) may be a manual correction entry reversing a duplicate write-off mistake.

**Status: Confirmed.** A563186 (14:51) and A563187 (14:52) are near-identical timestamps — clearly the same bad debt write-off entered twice by mistake. A563185 (14:50), the positive reversal, happened one minute *before* both of them chronologically — meaning it wasn't a correction booked after the fact. Given all three happened within a 2-minute window, the most likely explanation is a rapid data-entry sequence — a duplicate write-off followed almost immediately by an offsetting entry. Confirmed as a real, understood anomaly (0.0006% of the dataset) with no bearing on the core business question — no further action needed.

### Duplicate Row Investigation

Checked for exact duplicate rows (all 8 columns matching) using GROUP BY + HAVING COUNT(*) > 1.

**Result:** 32,907 duplicate groups, touching 67,242 total rows (6.3% of the dataset). Max repeat count: 20 (a single line item repeated 20 times under one invoice).

**Hypothesis tested:** given this is a warehouse-fulfilled online retailer (not a till/POS system), duplicates are more likely system logging errors than a customer scanning an item repeatedly. Tested whether duplicates are typically followed by a correcting cancellation (same StockCode + Customer ID, invoice starting with "C", negative Quantity).

**Single-case test (Invoice 555524, StockCode 22698, Customer 16923, 20x repeat):** confirmed a match — Invoice C555529, same StockCode/Customer, Quantity -24. Note: -24 doesn't exactly match the 20-unit duplicate total, so even this "confirmed" case isn't a perfectly clean 1:1 reversal.

**Scaled test across all 32,907 groups:**

```sql
WITH duplicates AS (
  SELECT
    COUNT(*) AS duplicate_count,
    Invoice, StockCode, Description, Quantity, InvoiceDate, Price, `Customer ID`, Country
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
  GROUP BY Invoice, StockCode, Description, Quantity, InvoiceDate, Price, `Customer ID`, Country
  HAVING COUNT(*) > 1
)

SELECT COUNT(*) AS groups_with_matching_correction
FROM duplicates AS d
WHERE EXISTS (
  SELECT 1
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined` AS t
  WHERE t.StockCode = d.StockCode
    AND t.`Customer ID` = d.`Customer ID`
    AND STARTS_WITH(t.Invoice, 'C')
    AND t.Quantity < 0
)
```

**Result:** 1,599 of 32,907 groups (4.9%) have a plausible matching correction elsewhere in the table.

**Finding:** the "duplicate → correction" hypothesis does NOT generalize. Over 95% of duplicate groups have no obvious matching cancellation.

**Decision:** duplicate rows (6.3% of the dataset) are left in place for the revenue analysis rather than excluded or adjusted, since no reliable, generalizable correction mechanism was found. Documented as a known characteristic of the dataset.

### StockCode Validity Check

```sql
SELECT DISTINCT(StockCode) AS distinct_stock_code
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
WHERE NOT REGEXP_CONTAINS(StockCode, '^([0-9]{5}[A-Za-z]?|POST|DOT)$')
```

**Result:** 66 distinct codes (out of 5,305 total) don't fit the standard "5 digits, optional trailing letter" pattern or the known POST/DOT codes.

```sql
SELECT COUNT(*) AS total_rows
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
WHERE NOT REGEXP_CONTAINS(StockCode, '^([0-9]{5}[A-Za-z]?|POST|DOT)$')
```

**Result:** 3,898 rows (0.37% of the dataset).

**Categories found:**
- `DCGS...` family, `gift_0001_XX` codes, single letters — likely legitimate non-standard product/gift-card codes
- `POST`, `DOT`, `BANK CHARGES`, `AMAZONFEE`, `ADJUST`, `ADJUST2`, `TEST001`, `CRUK`, `PADS`, `GIFT` — non-product administrative/fee entries
- A couple of formatting outliers noted but not deeply investigated given scale (e.g. two-letter suffix codes like `79323LP`, trailing whitespace)

**Decision:** left in the dataset as-is; flagged for exclusion from any "revenue by product" breakdown specifically in the Analyze phase.

### Zero-Price Row Investigation: Beyond the 4,382-Row Pattern

The original 4,382-row anomaly (null Description + null Customer ID + Price = 0) accounted for only part of all zero-price rows. Checked the full gap:

```sql
SELECT
  COUNTIF(Price = 0) AS total_zero_price,
  COUNTIF(Price = 0 AND is_zero_value_entry = TRUE) AS caught_by_current_flag,
  COUNTIF(Price = 0 AND is_zero_value_entry = FALSE) AS missed_by_current_flag
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined_clean`
```

**Result:** 6,202 total zero-price rows; 4,382 caught by the original pattern; **1,820 missed entirely**.

**Breaking down the 1,820 gap:**

```sql
SELECT
  COUNTIF(Description IS NOT NULL AND `Customer ID` IS NULL) AS has_description_no_customer,
  COUNTIF(Description IS NULL AND `Customer ID` IS NOT NULL) AS no_description_has_customer,
  COUNTIF(Description IS NOT NULL AND `Customer ID` IS NOT NULL) AS has_both
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined_clean`
WHERE Price = 0 AND is_zero_value_entry = FALSE
```

**Result:** 1,749 have a description but no customer; 0 have no description but a customer; 71 have both.

**Sampling the dominant 1,749 group revealed two distinct patterns:**
- **Pattern A — write-off notes typed directly into Description**: `lost`, `damaged`, `discoloured`, `damaged?`, `missing`, `missing?`, `found again`, `wet and rotting`, `ebay sales`, `Dotcom sold as set` — same underlying idea as the null-pattern and "A"-prefix entries, just a third logging mechanism.
- **Pattern B — genuine free/promotional giveaways**: real product descriptions, real customer-sized quantities, Price = 0 (e.g. WOODLAND STICKERS, DOOR MAT AIRMAIL).

**Also found in this sample: a swapped-code data entry error, unrelated to the zero-price pattern itself.** Invoice 489464 has StockCode `21733` with Description `"85123a mixed"`; Invoice 489467 (one minute later) has StockCode `85123A` with Description `"21733 mixed"` — each row's Description contains the *other* row's StockCode. A genuine, isolated data entry mix-up, not part of either Pattern A or B, but relevant later when it surfaced again as a bad `ANY_VALUE()` pick in the Top 10 Products query.

**Quantifying Pattern A** — tested nine adjustment keywords against the 1,749-row gap using `LIKE '%keyword%'`:

```sql
SELECT
  COUNTIF(Description LIKE '%lost%') AS total_lost,
  COUNTIF(Description LIKE '%damaged%') AS total_damaged,
  COUNTIF(Description LIKE '%missing%') AS total_missing,
  COUNTIF(Description LIKE '%found%') AS total_found,
  COUNTIF(Description LIKE '%discoloured%') AS total_discoloured,
  COUNTIF(Description LIKE '%wet%') AS total_wet,
  COUNTIF(Description LIKE '%rotting%') AS total_rotting,
  COUNTIF(Description LIKE '%sold%') AS total_sold,
  COUNTIF(Description LIKE '%sales%') AS total_sales
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined_clean`
WHERE Price = 0 AND is_zero_value_entry = FALSE AND Description IS NOT NULL AND `Customer ID` IS NULL
```

**Result:** 10, 91, 38, 31, 2, 22, 2, 44, 12 respectively (allowing overlap) — roughly 14% of the 1,749-row gap explained by clear write-off language; the remaining ~86% are genuine Pattern B free-item giveaways.

**Decision:** rather than expand keyword-matching indefinitely chasing 100% coverage (diminishing returns — e.g. a separately-found "given away" placeholder wasn't in the original nine keywords, affects £0 revenue, not worth chasing further), built two flags to cover the full zero-price picture: `is_write_off_note` (Price = 0 AND Description matches the adjustment-keyword pattern) and `is_zero_price_other` (Price = 0 AND neither `is_zero_value_entry` nor `is_write_off_note` apply) — a clean catch-all so every zero-price row lands in exactly one category, nothing left unaccounted for.

### Building the Clean, Reusable Table

Combined every cleaning finding into one table, using a flag-column design rather than hard exclusions — keeps all rows available for different future analyses.

```sql
CREATE OR REPLACE TABLE `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions` AS

WITH country_corrected AS (
  SELECT
    *,
    CASE
      WHEN Country = 'EIRE' THEN 'Ireland'
      WHEN Country = 'RSA' THEN 'South Africa'
      WHEN Country = 'Channel Islands' THEN 'United Kingdom'
      ELSE Country
    END AS Country_standardized
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.transactions_combined`
),

flagged AS (
  SELECT
    Invoice,
    StockCode,
    UPPER(StockCode) AS StockCode_standardized,
    Description,
    Quantity,
    InvoiceDate,
    SAFE.PARSE_TIMESTAMP('%m/%d/%y %H:%M', InvoiceDate) AS InvoiceDate_parsed,
    Price,
    `Customer ID`,
    Country,
    Country_standardized,
    (Country != Country_standardized) AS country_was_corrected,
    Country IN ('West Indies', 'Unspecified', 'European Community') AS country_not_geo_mappable,
    STARTS_WITH(Invoice, 'C') AS is_cancellation,
    (STARTS_WITH(Invoice, 'A') AND LENGTH(Invoice) = 7) AS is_bad_debt_adjustment,
    (Description IS NULL AND `Customer ID` IS NULL AND Price = 0) AS is_zero_value_entry,
    (Price = 0 AND COALESCE(REGEXP_CONTAINS(LOWER(Description), 'lost|damaged|missing|found|discoloured|wet|rotting|sold|sales'), FALSE)) AS is_write_off_note,
    NOT REGEXP_CONTAINS(StockCode, '^([0-9]{5}[A-Za-z]?)$') AS is_non_product_code,
    COUNT(*) OVER (
      PARTITION BY Invoice, StockCode, Description, Quantity, InvoiceDate, Price, `Customer ID`, Country
    ) > 1 AS is_duplicate_row
  FROM country_corrected
)

SELECT
  *,
  (Price = 0 AND NOT is_zero_value_entry AND NOT is_write_off_note) AS is_zero_price_other
FROM flagged
```

*(Note: this is the final version, after two later corrections documented in Analyze — dropping the POST/DOT whitelist from `is_non_product_code`, and adding `StockCode_standardized` to fix a casing-split bug. Shown here in final form to avoid presenting a table that gets rebuilt twice in the log.)*

**Country mapping decisions:**
- `EIRE` → `Ireland`, `RSA` → `South Africa`: exact corrections, not approximations
- `Channel Islands` → `United Kingdom`: reasonable approximation, flagged via `country_was_corrected`
- `West Indies`, `Unspecified`, `European Community`: no single country can accurately represent these — flagged via `country_not_geo_mappable` instead

**Verification 1 — row count:** 1,067,371 — matches `transactions_combined` exactly.

**Verification 2 — null-safety check on all derived columns:** all zeros. Table fully verified and ready to build on.

### Flag Accuracy Audit

Beyond the null-safety check, tested whether flags are semantically accurate, not just non-null.

**Bug found and fixed:** `REGEXP_CONTAINS` and `AND` both propagate `NULL` (not `FALSE`) when `Description` is null, silently breaking `is_write_off_note` for the 4,382 rows with null Description. Fixed with `COALESCE(..., FALSE)`. Re-verified: 0 nulls across all 9 boolean columns after the fix.

**`is_cancellation` accuracy check:** among 19,494 "C"-prefixed invoices, checked for any with non-negative Quantity — this also serves as the closeout for the original checklist question "do negative Quantity rows align with cancelled invoices?"

**Result:** 1 exception — Invoice C496350, StockCode "M", Description "Manual", Quantity 1, Price 373.57. A staff manual entry using a "C" invoice number, not a genuine product cancellation. 19,493 of 19,494 (99.995%) align correctly.

**Decision:** left as-is, no new flag needed — negligible and well-understood.

**`is_duplicate_row` accuracy check:** cross-validated the window function count against the original GROUP BY/HAVING duplicate count computed earlier, before Country standardization and flag columns were added.

**Result:** 67,242 — exact match. Confirms the window function is accurate and stable through all subsequent transformations.

## ANALYZE

### Country Standardization: Scope of Use

Two different types of Country renaming were applied when building `cleaned_transactions`:
- **EIRE → Ireland, RSA → South Africa** — exact corrections. Safe to use everywhere.
- **Channel Islands → United Kingdom** — a deliberate approximation, applied specifically so the country renders correctly on a Looker Studio geo-map.

**Verification:** quantified the impact of the Channel Islands remapping on revenue totals.

```sql
SELECT
  total,
  channel_island_revenue,
  (channel_island_revenue / total) * 100 AS percentage
FROM (
  SELECT
    SUM(Price * Quantity) AS total,
    SUM(CASE WHEN Country = 'Channel Islands' THEN Price * Quantity ELSE 0 END) AS channel_island_revenue
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
) AS revenue_summary
```

**Result:** total revenue £19,287,250.57; Channel Islands revenue £41,453.61 — 0.21% of the total. Negligible.

**Decision:** `Country_standardized` is reserved specifically for the Looker Studio map visual. All other analysis uses the original `Country` column.

### Investigating December Revenue Anomalies

Dec 2011 revenue (£433,686) looked anomalously low compared to every other month — investigated before trusting it in the trend chart.

**Dec 2011:** confirmed data ends 2011-12-09 12:50 UTC — matches the dataset's documented end date exactly. This is a partial month, not a real decline.

**Dec 2009 (checked as a precaution, since it's also a dataset boundary):** confirmed data runs 2009-12-01 07:45 to 2009-12-23 16:58.

**Ruled out as a data gap by comparing against Dec 2010** (a fully interior, non-boundary month): confirmed data also runs only to 2010-12-23 17:41 — the exact same cutoff date, one year apart. A genuine, recurring business pattern (holiday order cutoff around Dec 23rd), not missing data.

**Decision:** Dec 2009 and Dec 2010 are both complete, legitimate months — kept as-is. Dec 2011 is genuinely partial — kept in the trend chart, with a clear footnote/annotation noting it only reflects 9 days of data.

### Query #1: Monthly Revenue Trend

```sql
SELECT
  DATE_TRUNC(InvoiceDate_parsed, MONTH) AS month,
  SUM(Price * Quantity) AS revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
GROUP BY month
ORDER BY month
```

**Finding:** clear seasonality — November peaks both years (pre-Christmas buildup), February troughs. See December anomalies investigation above for the two boundary-month caveats.

### Query #2: Cancellation Impact — Top 10 Countries by Revenue

Confirmed top 10 countries represent 97.5% of total revenue (£18,805,076.58 of £19,287,250.57) — extreme market concentration, worth noting as a standalone finding.

```sql
SELECT
  Country,
  total_revenue,
  cancelled_revenue,
  (cancelled_revenue / total_revenue) * 100 AS cancellation_percentage
FROM (
  SELECT
    Country,
    SUM(Price * Quantity) AS total_revenue,
    SUM(CASE WHEN is_cancellation = TRUE THEN Price * Quantity ELSE 0 END) AS cancelled_revenue
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  GROUP BY Country
  ORDER BY total_revenue DESC
  LIMIT 10
) AS top_10_with_cancellations
ORDER BY total_revenue DESC
```

**Result:**

| Country | Revenue | Cancelled | Cancellation % |
|---|---|---|---|
| United Kingdom | 16,382,583.90 | -1,329,717.74 | -8.12% |
| EIRE | 615,519.55 | -48,912.23 | -7.95% |
| Netherlands | 548,524.95 | -5,707.39 | -1.04% |
| Germany | 417,988.56 | -13,273.90 | -3.18% |
| France | 328,191.80 | -28,752.80 | -8.76% |
| Australia | 167,129.07 | -2,839.04 | -1.70% |
| Switzerland | 99,728.76 | -1,282.53 | -1.29% |
| Spain | 91,859.48 | -17,319.05 | **-18.85%** |
| Sweden | 87,809.42 | -4,094.30 | -4.66% |
| Denmark | 65,741.09 | -4,121.10 | -6.27% |

**Finding:** cancellation rate varies significantly, from ~1% (Netherlands, Switzerland) to ~19% (Spain) — a clear outlier worth investigating.

### Investigating Spain's Cancellation Rate (-18.85%)

```sql
SELECT
  Invoice, StockCode, Description, Quantity, Price,
  (Price * Quantity) AS cancelled_amount, InvoiceDate
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
WHERE Country = 'Spain' AND is_cancellation = TRUE
ORDER BY cancelled_amount ASC
```

**Finding:** the -£17,319 total is not spread across many ordinary cancellations. ~71% comes from two large bulk-order invoices (C527790, C531557); ~23% from "Manual" staff-entered adjustments. ~94% traces back to a small handful of events, not broad customer-level cancellation behavior.

**Decision:** the -18.85% figure is accurate and kept as-is. Footnote: *"Spain's cancellation rate is substantially driven by two large wholesale-order cancellations rather than broad customer behavior — worth flagging for follow-up investigation, though outside this analysis's scope."*

### Query #3: Bottom 10 Countries by Revenue

```sql
SELECT
  Country,
  SUM(Price * Quantity) AS total_revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
GROUP BY Country
ORDER BY total_revenue ASC
LIMIT 10
```

**Result:** Nigeria (-£6.66), Saudi Arabia (£131.17), West Indies (£536.41), Czech Republic (£707.72), Korea (£949.82), Bermuda (£1,253.14), European Community (£1,291.75), Brazil (£1,411.87), Lebanon (£1,865.91), Bahrain (£2,861.55).

**Decision — non-country buckets:** European Community and West Indies are flagged `country_not_geo_mappable = TRUE`, not single countries — plus Nigeria's genuine net-negative revenue. Kept unfiltered, matching the business's own system and real recorded numbers. In the Looker Studio visual, non-country entries are visually distinguished rather than excluded — consistent with the flag-column philosophy used throughout: never silently drop data, make distinctions visible instead.

### Fixing is_non_product_code: POST/DOT Were Being Whitelisted Incorrectly

Found while building the Top 10 Products query: POST and DOT appeared in the ranking despite the `is_non_product_code = FALSE` filter.

**Root cause:** the original regex `'^([0-9]{5}[A-Za-z]?|POST|DOT)$'` explicitly whitelisted POST and DOT — appropriate for the original data-quality audit, incorrect for a "is this a sellable product" filter.

**Fix:** simplified the pattern to `'^([0-9]{5}[A-Za-z]?)$'` — dropping the whitelist. Re-verified: row count and null-safety checks unchanged. *(Table definition above already reflects this fix.)*

### Fixing StockCode Case Inconsistency

Found while reviewing the ≥20-orders bottom-10 product list: `84993A` and `84993a` were being treated as two separate products (177 orders/£660.79 vs. 30 orders/£42.71).

```sql
SELECT
  UPPER(StockCode) AS stockcode_upper,
  COUNT(DISTINCT StockCode) AS casing_variants,
  ANY_VALUE(Description) AS sample_description
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
WHERE is_non_product_code = FALSE
GROUP BY stockcode_upper
HAVING COUNT(DISTINCT StockCode) > 1
```

**Result:** 171 products affected — a widespread issue, not an isolated case.

**Fix:** added `StockCode_standardized = UPPER(StockCode)` — original `StockCode` preserved for traceability. *(Table definition above already reflects this fix.)* Re-verified: row count, null-safety, and correct merge of `84993A` (~207 orders, ~£703.50) all confirmed.

### Query #4: Top 10 Products by Revenue

First attempt used `ANY_VALUE(Description)` for the product name — this surfaced two problems, corrected below. Final, correct version:

```sql
WITH description_frequency AS (
  SELECT StockCode_standardized, Description, COUNT(*) AS description_count
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  WHERE is_non_product_code = FALSE
  GROUP BY StockCode_standardized, Description
),
most_common_description AS (
  SELECT StockCode_standardized, Description,
    ROW_NUMBER() OVER (PARTITION BY StockCode_standardized ORDER BY description_count DESC) AS rank
  FROM description_frequency
  QUALIFY rank = 1
)
SELECT
  t.StockCode_standardized,
  d.Description AS product_description,
  SUM(t.Price * t.Quantity) AS total_revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions` AS t
JOIN most_common_description AS d ON t.StockCode_standardized = d.StockCode_standardized
WHERE t.is_non_product_code = FALSE
GROUP BY t.StockCode_standardized, d.Description
ORDER BY total_revenue DESC
LIMIT 10
```

**Result:**

| StockCode | Product | Revenue |
|---|---|---|
| 22423 | REGENCY CAKESTAND 3 TIER | 327,813.65 |
| 85123A | WHITE HANGING HEART T-LIGHT HOLDER | 257,712.41 |
| 85099B | JUMBO BAG RED RETROSPOT | 183,390.15 |
| 47566 | PARTY BUNTING | 147,948.50 |
| 84879 | ASSORTED COLOUR BIRD ORNAMENT | 131,413.85 |
| 22086 | PAPER CHAIN KIT 50'S CHRISTMAS | 121,662.14 |
| 79321 | CHILLI LIGHTS | 84,854.16 |
| 22197 | SMALL POPCORN HOLDER | 80,300.07 |
| 22386 | JUMBO BAG PINK POLKADOT | 76,244.93 |
| 84347 | ROTATING SILVER ANGELS T-LIGHT HLDR | 73,814.72 |

**Fixing ANY_VALUE's unreliable descriptions:** the first version pulled `"21733 mixed"` for `85123A` (the garbled swap anomaly found earlier) and `"?"` for `47566` (PARTY BUNTING).

**Scope check on "?":** 92 rows across 88 distinct products — a thin, scattered placeholder pattern.

**Root cause:** `ANY_VALUE()` picks an arbitrary row per group, with no preference for the genuine common description over a rare placeholder.

**Fix:** replaced `ANY_VALUE(Description)` with a frequency-ranked lookup (`ROW_NUMBER() ... QUALIFY rank = 1`), joined back to the revenue calculation — shown in the final query above. Re-verified: revenue figures unchanged; `47566` now correctly shows PARTY BUNTING; `85123A`'s revenue also corrected upward (£253,720.02 → £257,712.41) once the StockCode-casing fix merged in its lowercase variant.

### Net-Negative Products — and a Correction on What This Actually Measures

```sql
SELECT
  COUNT(*) AS negative_product_count,
  SUM(total_revenue) AS total_negative_revenue
FROM (
  SELECT
    StockCode_standardized,
    SUM(Price * Quantity) AS total_revenue
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  WHERE is_non_product_code = FALSE
  GROUP BY StockCode_standardized
) AS product_totals
WHERE total_revenue < 0
```

**Result:** 22 products, totaling -£519.88.

**Correction 1:** this is a revenue metric, not a "money lost" metric — no cost-of-goods data exists to calculate true financial loss.

**Correction 2 — checked whether zero-price cancellations were invisible to this and the cancellation-impact figures:**

```sql
SELECT
  COUNTIF(is_cancellation = TRUE) AS total_cancelled_rows,
  COUNTIF(is_cancellation = TRUE AND Price = 0) AS zero_price_cancelled_rows,
  (COUNTIF(is_cancellation = TRUE AND Price = 0) / COUNTIF(is_cancellation = TRUE)) * 100 AS percentage
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
```

**Result:** 19,494 total cancelled rows; 0 with Price = 0 (0.0%).

**Finding:** the zero-price write-off pattern never overlaps with formal "C" cancellations — two entirely separate mechanisms. The cancellation-impact figures are NOT missing any hidden cancelled value.

**Decision:** cancellation-impact findings stand as accurate and complete. The net-negative-products figure remains a minor, non-material finding.

### Query #5: Bottom 10 Products by Revenue (≥20 orders)

```sql
WITH description_frequency AS (
  SELECT StockCode_standardized, Description, COUNT(*) AS description_count
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  WHERE is_non_product_code = FALSE
  GROUP BY StockCode_standardized, Description
),
most_common_description AS (
  SELECT StockCode_standardized, Description,
    ROW_NUMBER() OVER (PARTITION BY StockCode_standardized ORDER BY description_count DESC) AS rank
  FROM description_frequency
  QUALIFY rank = 1
)
SELECT
  t.StockCode_standardized,
  d.Description AS product_description,
  COUNT(DISTINCT t.Invoice) AS order_count,
  SUM(t.Price * t.Quantity) AS total_revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions` AS t
JOIN most_common_description AS d ON t.StockCode_standardized = d.StockCode_standardized
WHERE t.is_non_product_code = FALSE
GROUP BY t.StockCode_standardized, d.Description
HAVING COUNT(DISTINCT t.Invoice) >= 20
ORDER BY total_revenue ASC
LIMIT 10
```

**Threshold rationale:** tested ≥5 and ≥20 orders side by side. ≥5 was still contaminated by anomaly patterns already investigated separately (net-negative products, "given away"). ≥20 produced a genuinely clean list.

**Result:**

| Product | Orders | Revenue |
|---|---|---|
| BLACK CHERRY LIGHTS | 111 | £15.25 |
| ASSORTED COLOUR JUMBO PEN | 28 | £29.32 |
| MULTICOLOUR CRUSOE CHECK LAMPSHADE | 20 | £30.66 |
| PINK/GREEN FLOWER DESIGN BIG MUG | 21 | £30.81 |
| PORCELAIN CHERUB BELL SMALL | 29 | £31.98 |
| LETTER "V" BLING KEY RING | 24 | £32.33 |
| WINE BOTTLE DRESSING DARK BLUE | 29 | £39.08 |
| LETTER "F" BLING KEY RING | 27 | £39.23 |
| WINE BOTTLE DRESSING LT.BLUE | 26 | £39.87 |
| PET MUG, GOLDFISH | 30 | £47.12 |

**Finding:** BLACK CHERRY LIGHTS stands out — 111 orders but only £15.25 total revenue, an extremely low price per unit. Candidate for pricing review.

**Verified applying the same ANY_VALUE fix here as a precaution:** identical result — descriptions were already reliable for this low-volume set, but worth confirming rather than assuming.

## SHARE

### Turning Validated Queries into Reusable BigQuery Views

Every chart in the dashboard is backed by a named View built from a query already verified against real output earlier in this log.

```sql
CREATE OR REPLACE VIEW `project-ee6309fa-8dfb-435e-920.online_retail_combine.v01_revenue_trend_monthly` AS
SELECT
  DATE_TRUNC(InvoiceDate_parsed, MONTH) AS month,
  SUM(Price * Quantity) AS revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
GROUP BY month
ORDER BY month;

CREATE OR REPLACE VIEW `project-ee6309fa-8dfb-435e-920.online_retail_combine.v02_top10_countries_cancellation` AS
SELECT
  Country, total_revenue, cancelled_revenue,
  (cancelled_revenue / total_revenue) * 100 AS cancellation_percentage
FROM (
  SELECT Country,
    SUM(Price * Quantity) AS total_revenue,
    SUM(CASE WHEN is_cancellation = TRUE THEN Price * Quantity ELSE 0 END) AS cancelled_revenue
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  GROUP BY Country ORDER BY total_revenue DESC LIMIT 10
) AS top_10_with_cancellations
ORDER BY total_revenue DESC;

CREATE OR REPLACE VIEW `project-ee6309fa-8dfb-435e-920.online_retail_combine.v03_bottom10_countries` AS
SELECT Country, SUM(Price * Quantity) AS total_revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
GROUP BY Country ORDER BY total_revenue ASC LIMIT 10;

CREATE OR REPLACE VIEW `project-ee6309fa-8dfb-435e-920.online_retail_combine.v04_top10_products` AS
WITH description_frequency AS (
  SELECT StockCode_standardized, Description, COUNT(*) AS description_count
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  WHERE is_non_product_code = FALSE
  GROUP BY StockCode_standardized, Description
),
most_common_description AS (
  SELECT StockCode_standardized, Description,
    ROW_NUMBER() OVER (PARTITION BY StockCode_standardized ORDER BY description_count DESC) AS rank
  FROM description_frequency QUALIFY rank = 1
)
SELECT t.StockCode_standardized, d.Description AS product_description,
  SUM(t.Price * t.Quantity) AS total_revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions` AS t
JOIN most_common_description AS d ON t.StockCode_standardized = d.StockCode_standardized
WHERE t.is_non_product_code = FALSE
GROUP BY t.StockCode_standardized, d.Description
ORDER BY total_revenue DESC LIMIT 10;

CREATE OR REPLACE VIEW `project-ee6309fa-8dfb-435e-920.online_retail_combine.v05_bottom10_products_min20orders` AS
WITH description_frequency AS (
  SELECT StockCode_standardized, Description, COUNT(*) AS description_count
  FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions`
  WHERE is_non_product_code = FALSE
  GROUP BY StockCode_standardized, Description
),
most_common_description AS (
  SELECT StockCode_standardized, Description,
    ROW_NUMBER() OVER (PARTITION BY StockCode_standardized ORDER BY description_count DESC) AS rank
  FROM description_frequency QUALIFY rank = 1
)
SELECT t.StockCode_standardized, d.Description AS product_description,
  COUNT(DISTINCT t.Invoice) AS order_count, SUM(t.Price * t.Quantity) AS total_revenue
FROM `project-ee6309fa-8dfb-435e-920.online_retail_combine.cleaned_transactions` AS t
JOIN most_common_description AS d ON t.StockCode_standardized = d.StockCode_standardized
WHERE t.is_non_product_code = FALSE
GROUP BY t.StockCode_standardized, d.Description
HAVING COUNT(DISTINCT t.Invoice) >= 20
ORDER BY total_revenue ASC LIMIT 10;
```

**Note on saved queries vs. Views:** BigQuery's "saved query" feature only bookmarks SQL text for personal reuse — not a real database object, can't be connected to by external tools. Views are live objects any tool can connect to directly.

### Building the Looker Studio Dashboard

**Debugging log — real issues hit and resolved:**

- **Dec 2009 initially appeared as zero revenue.** Root cause: purely visual — the bar sat flush against the chart's left edge, clipped from view. Fixed by widening the default date range beyond the data's actual boundaries.
- **"Invalid dimension" error on the cancellation chart.** Root cause: at the data source level, all three numeric fields were auto-classified as Dimensions instead of Metrics. Fixed by rebuilding the field as a proper calculated field.
- **Same error, different cause.** Chart type was auto-suggested as Time Series, which requires a date dimension; `Country` is text. Fixed by explicitly selecting Bar chart.
- **Negative bars rendering downward** — confirmed correct, expected behavior, not a bug.
- **Dual-axis distortion** on Bottom 10 Products (`total_revenue` and `order_count` on non-comparable scales). Resolved by dropping `order_count` as a plotted metric — preserved in the shared Notes instead.
- **Long product/country names truncating.** Resolved by converting the two product charts to Table type and Bottom 10 Countries to horizontal bar.
- **Recurring rendering glitches** — resolved by reloading; confirmed not real data issues via cross-checking against known-correct BigQuery results.

**Design decisions:**
- Report-level theme: blue (`#2A78D6`) for revenue/magnitude, red (`#E34948`) manually overridden on the cancellation chart to visually distinguish loss from magnitude.
- One shared Notes text box covering the Dec 2011 caveat, Spain's cancellation context, the European Community/West Indies non-country caveat, and the ≥20-order filter rationale.
- Bottom-performer charts kept genuinely unfiltered rather than excluded — consistent with the flag-column philosophy: never silently drop data, make distinctions visible instead.

**Final dashboard structure:** Revenue Trend (full width, top) → Cancellation Rate by Market + Bottom 10 Countries (row 2) → Top 10 Products + Bottom 10 Products (row 3) → shared Notes (bottom).